# Exp1034 — DNA A260 screening LOD on the FilterMax F5

**Fresh preparation after evaporation:** reset the physical starting deck, restart the notebook kernel, and run the cells from the top to create a new `ExperimentSession` and output directory. Do not replay Friday's session or use its plate-resume plan. Its records remain preserved in `runs/physical-20261004T001957Z-fab79d55`.

Start with A30 at 3000 µL total and measured 16 ng/µL, A31/A32 at 4000 µL clean matching buffer each, empty A29–A25 and plate, replenished tips, and no attached old tip. All 96 wells—including A9—belong to this fresh run; no diagnostic withdrawal is subtracted.

Run the cells in order. **The notebook defaults to simulation** using the same LiquidHandler calls as the physical experiment. The operator confirmed the A25 300 µL tip reach test; the complete plate dispensing method is still under evaluation.

One twofold plasmid-DNA dilution chain supplies 16, 8, 4, 2, 1 and 0.5 ng/µL; final volumes are 200, 100, 75 and 50 µL. Each volume has six blanks and three **dispensing replicates** per concentration. The supplied 96-well map and replicate labels are preserved.

**Corrected deck:** bare `hamilton_tube_carrier_32_a00` at track 19; `hamilton_mfx_carrier_L5_base` with five `hamilton_mfx_plateholder_DWP_metal_tapped` modules at track 7; Corning **3635** at module 0 using the user-confirmed `Cor_96_wellplate_360ul_Fb` geometry; `TIP_CAR_480_A00` at track **25**, with 1000 µL filtered / 300 µL / 50 µL racks at positions 0 / 1 / 2. A32→A25 occupy tube sites 0→7, front to back.

The tube definition is copied from branch `add-12x75mm-test-tube`, commit `d62db0a42`, in `mckesson_tubes.py`; tube mounting and base thickness follow the supplied Exp1031 notebook. No checkout switch or merge is required. Keep this notebook and its Python/JSON sidecars together.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import sys
import uuid

EXPERIMENT_DIR = Path(
  "/home/hamilton-robot/Documents/hamilton-startlet-scripts/Exp1034--Filtermax-LOD"
)
PLR_CHECKOUT = Path("/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot")
sys.path.insert(0, str(PLR_CHECKOUT))
sys.path.insert(0, str(EXPERIMENT_DIR))

In [ ]:
import pandas as pd
from IPython.display import display
from exp1034 import manifest, environment
import importlib
import notebook_protocol

importlib.reload(notebook_protocol)
from notebook_protocol import ExperimentSession, Settings

print("PyLabRobot:", environment()["pylabrobot_file"])
print(
  "PLR version / commit:", environment()["pylabrobot_version"], environment()["pylabrobot_commit"]
)

## 1. Run inputs

Set `number_of_tube_mixes` at the top of the Settings cell to an **integer from 1 to 9**. One cycle aspirates and dispenses 750 µL within the same tube. Each receiving DNA tube is mixed this many times **after both 750 µL transfers** are complete; initial A30 mixing uses the same count. Rerun the entire Settings cell before starting a fresh session to apply a change. This cell reloads `notebook_protocol.py` and imports fresh `Settings` and `ExperimentSession` definitions, so an older kernel definition cannot supply the 1.0 mm preparation floor. The preparation floor is 4.3 mm and is printed with the inputs. Use this cell only for a fresh run; it creates a new session, not a continuation of a completed run.

A30 must contain **3000 µL TOTAL at a measured 16 ng/µL**, not 3 mL buffer plus an unaccounted DNA addition. A32 and A31 start with 4000 µL matching clean buffer each; A29–A25 and the plate start empty.

The values below are **candidate method settings**. The 200 µL residual floor plus 25 µL total loss allowance consumes the 225 µL allowance for A30–A26. These settings are for method development. Physical accessibility at low volume, tip/rim clearance, noncontact dispensing, bubble control and homogeneity at the selected mixing count remain to be established. The source height model is the branch's measured linear fit; its nonzero intercept does not establish dead volume.

For this fresh physical run, set `simulate=False`, enter the actual measurement in `measured_a30_concentration`, and fill in the operator. The method reference remains optional; a reach test does not establish mixing or dispensing accuracy. The design target is fixed at 16 ng/µL; it does not fill in the measurement. If A30 was measured at 16 ng/µL, replace `None` with `16.0`. After editing inputs, rerun the **entire Settings cell below** so that `session` uses them; editing the text alone does not update an existing session. A fresh typed approval is required before STARBackend setup, following the repository rule: **“Never drive real hardware without explicit, per-run approval.”** No need to reconfirm the corrected carrier identities or plate dimensions supplied in this task.

`method_validation_reference` is **optional**. Leave it blank for an exploratory first run; the approval prompt and run record will state that no prior method validation was supplied. If prior checks exist, enter a notebook page, file path, or factual notes. The software still checks the concentration, operator, planned volumes, tip capacity and modeled clearance before hardware operation.

All preparation transfers use **250 µL/s for both aspiration and dispense**, including contact buffer prefills and noncontact DNA transfers. Mixing explicitly uses **400 µL/s in both directions**, 750 µL per cycle, with the count set by `number_of_tube_mixes` (default 8). Plate transfer rates are set separately below and displayed in the geometry review, using 300 µL tips.

Buffer prefills finish **1 mm below** the modeled final surface. The planner uses the McKesson tube's height function, giving 11.44 mm after 750 µL and 20.25 mm after 1500 µL, above the inner bottom. The tip contacts liquid near the end of each addition. Preparation DNA transfers remain noncontact.

Preparation flow rates are explicitly set to 250 µL/s in both directions. Automatic volume correction stays disabled; prewetting, transport air, blowout air and over-aspiration remain zero. The resolved rates and heights are displayed before instrument setup.
Plate-tip heights use the verified settings from `Exp1034--Plate-only.ipynb`: every plate dispense is **2.50 mm above the modeled inner well bottom**, independent of the 50, 75, 100 or 200 µL volume. Tube aspirations during plate filling descend **an additional 4 mm** below the original calculated path, with the existing **3 mm inner-bottom/reach limit** and 2 mm tip/rim margin. Both the initial aspiration height and its surface-following endpoint respect that limit. Every preparation tube-to-tube aspiration, including clean buffer and both 750 µL aliquots of each DNA transfer, also descends **an additional 4 mm** at the start and surface-following endpoint. With 1000 µL preparation tips, endpoints stop at the configured **4.3 mm preparation aspiration floor** if necessary; the last withdrawals from A31 and A32 reach this limit. This floor limits the requested lowering; all DNA transfers retain the full 4 mm adjustment. Mixing remains at 6 mm. Tube dispense heights retain the buffer surface and DNA noncontact settings described above.


In [ ]:
number_of_tube_mixes = 1  # integer from 1 to 9; cycles per tube

# Refresh the protocol before creating a fresh session.
import importlib
import notebook_protocol

importlib.reload(notebook_protocol)
from notebook_protocol import ExperimentSession, Settings

settings = Settings(
  simulate=False,
  channel=3,  # individual channel, zero-based
  measured_a30_concentration=16,  # actual measurement in ng/µL; must be 16.0 for a physical run
  operator="Harley",
  method_validation_reference="",  # optional; leave blank for this first run
  source_dead_volume=200.0,  # candidate µL; validate physically
  source_total_loss_allowance=25.0,  # total allowance per source, not per aspiration
  mix_volume=750.0,
  mix_cycles=number_of_tube_mixes,
  mix_height=6.0,  # mm above cavity bottom, Exp1031 aspiration-height reference
  mix_flow=400.0,  # µL/s, both aspiration and dispense during mixing
  preparation_aspirate_flow=250.0,  # µL/s, all preparation transfers
  preparation_dispense_flow=250.0,  # µL/s, both buffer and DNA transfers
  preparation_aspiration_lowering=4.0,  # mm lower for every buffer/DNA tube transfer; clearance-limited
  preparation_minimum_tip_height=4.3,  # mm above inner tube bottom; preparation aspiration floor
  buffer_prefill_submersion=1.0,  # mm below the final modeled surface, buffer prefills only
  aspirate_flow=100.0,  # µL/s, plate aspiration only
  dispense_flow=120.0,  # µL/s, plate dispense only
  plate_aspiration_lowering=4.0,  # verified Plate-only setting; limited by bottom/tip clearance
  plate_dispense_height=2.50,  # fixed mm above each inner well bottom, independent of volume
  immersion=2.0,
  minimum_tip_height=1.0,
  noncontact_clearance=3.0,
  tip_rim_margin=2.0,
  traverse_height=247.6,
  settling_time=1.0,
)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
output_dir = (
  EXPERIMENT_DIR / "runs" / (("simulation-" if settings.simulate else "physical-") + run_id)
)
session = ExperimentSession(settings, output_dir)
print("Mode:", "SIMULATION" if settings.simulate else "PHYSICAL RUN — approval required")
print("Measured A30 concentration (ng/µL):", session.settings.measured_a30_concentration)
print("Outputs:", output_dir)
print("Tip counts:", session.report["tip_counts"])
print("Mixing cycles per tube:", session.settings.mix_cycles)
print("Preparation minimum aspiration height (mm):", session.settings.preparation_minimum_tip_height)
display(pd.DataFrame(session.ledger))

## 2. Review geometry, heights and manifest — still no instrument connection

The plate uses **seven 300 µL nonfiltered tips**, one for all blanks and one for each DNA concentration. Preparation uses five 1000 µL filtered tips. Every well receives its original 50, 75, 100 or 200 µL volume with no change to the well map.

The operator confirmed that the 300 µL B1 tip reached 1 mm above A25's inner bottom, aspirated 300 µL and retracted successfully. For this deck and these McKesson tubes, the plan uses the tested 73 mm insertion limit, retaining a 2 mm margin. The deepest plate aspiration is limited to 71 mm insertion, retaining the 2 mm margin within the tested 73 mm reach. Tip length and capacity are unchanged; the physical check establishes reach, not dispensing accuracy or mixing performance.

A1 is rear-left on the STAR deck. The 3635 plate geometry is user-confirmed. The 50 µL group remains an experimental volume for this plate.

In [ ]:
print(session.setup.deck.summary())
display(pd.DataFrame.from_dict(session.setup.audit["tube_coordinates"], orient="index"))
geometry_rows = [
  {
    key: event.get(key)
    for key in (
      "step",
      "phase",
      "action",
      "source",
      "destination",
      "volume",
      "tip_size",
      "aspiration_height",
      "aspiration_end_height",
      "aspiration_height_limited",
      "aspiration_lowering_requested",
      "aspiration_start_lowering_applied",
      "aspiration_end_lowering_applied",
      "aspiration_minimum_height",
      "dispense_height",
      "destination_inner_bottom_z",
      "dispense_tip_z",
      "destination_after",
      "aspirate_flow_rate",
      "dispense_flow_rate",
      "dispense_jet",
      "tip_penetration_at_end",
    )
  }
  for event in session.plan
  if event["action"] in ("transfer", "mix_cycle")
]
geometry_df = pd.DataFrame(geometry_rows)
print("Resolved transfer/mixing flow rates (µL/s):")
display(geometry_df[[
  "phase", "action", "tip_size", "aspirate_flow_rate", "dispense_flow_rate", "dispense_jet"
]].drop_duplicates().reset_index(drop=True))
preparation_geometry = geometry_df[
  (geometry_df["phase"] == "preparation") & (geometry_df["action"] == "transfer")
]
print(f"Preparation tube aspiration lowering: {session.settings.preparation_aspiration_lowering:g} mm; "
      f"minimum inner-bottom height: {session.settings.preparation_minimum_tip_height:g} mm.")
print("Preparation aspirations limited by bottom/tip clearance:",
      int(preparation_geometry["aspiration_height_limited"].sum()))
display(preparation_geometry)
plate_geometry = geometry_df[
  (geometry_df["phase"] == "plate") & (geometry_df["action"] == "transfer")
]
print("Plate heights: fixed 2.50 mm above the inner well bottom; tube aspirations lowered 4 mm.")
print("Plate aspirations limited by bottom/tip clearance:", int(plate_geometry["aspiration_height_limited"].sum()))
display(plate_geometry[[
  "volume", "dispense_height", "destination_inner_bottom_z", "dispense_tip_z"
]].drop_duplicates().sort_values("volume").reset_index(drop=True))
print(
  "Every planned transfer and mix passed capacity, source-floor, reach and modeled height checks."
)

manifest_df = pd.DataFrame([asdict(record) for record in manifest()])
plate_map = pd.DataFrame(index=list("ABCDEFGH"), columns=range(1, 13))
for record in manifest():
  sample = f"{record.final_concentration:g} ng/µL" if record.final_concentration else "blank"
  plate_map.loc[record.well[0], int(record.well[1:])] = (
    f"{record.final_volume} µL | {sample} | rep{record.replicate} | {record.source_tube}"
  )
display(plate_map)
display(manifest_df)

## 3. Start the session

In simulation, this initializes only ChatterboxBackend and PLR trackers. For hardware, all physical inputs are checked **before** the `APPROVE EXP1034 RUN` prompt and instrument setup. Tracker initialization describes the verified starting deck; it never dispenses anything.

A run directory is created once. Do not rerun cells after a partial failure or reuse a partially filled deck as if empty. Hardware errors latch the session, log the last attempted action, and perform no automatic retry or recovery movement. An unacknowledged manual refill pauses at that checkpoint and can be retried without replaying preparation.

In [ ]:
await session.start()
print("Session initialized. Run the preparation cell next.")

## 4. Prefill and prepare the fresh dilution chain

One 1000 µL filtered tip prefills each empty A29–A25 with 750 µL from A32 plus 750 µL from A31. For each addition, the tip finishes 1 mm below the destination's final surface, calculated with the McKesson height-volume function. The two dispense heights are **11.44 mm** and **20.25 mm** above the inner bottom. These are contact surface dispenses (`jet=False`, `blow_out=True`). The tip is reused only while all contacted receiving tubes contain clean buffer; the planner rejects contact prefilling into a tube containing DNA.

The same tip then mixes A30. A30→A29 is two 750 µL **noncontact** deliveries, followed by mixing A29 and discarding the tip. Repeat the four remaining links with a fresh tip per link. DNA transfer heights remain above the destination's final liquid surface so the tip can return to the higher-concentration source between aliquots.

All 20 preparation tube-to-tube transfers use **4 mm extra aspiration lowering** at both the start and endpoint, with bottom/reach limits retained. The last 750 µL buffer withdrawals from A31 and A32 stop at 4.3 mm above the inner tube bottom. This changes aspiration heights only.

Preparation transfers use **250 µL/s aspiration and dispense** for both buffer and DNA; mixing uses **400 µL/s aspiration and dispense, 750 µL × `number_of_tube_mixes` cycles** at the existing 6 mm mixing height. No extra diagnostic aliquot is removed from A25. This cell runs only on the new session created above.

### Recover the first A25 buffer aspiration only

If preparation failed while aspirating the first 750 µL from A32 into A25, and you have confirmed that **no liquid was aspirated**, keep the original kernel, `session`, attached empty 1000 µL A1 tip and deck. Do not rerun Settings or setup. In a separate cell, run:

```python
import importlib
import notebook_protocol

importlib.reload(notebook_protocol)
await notebook_protocol.resume_preparation(session, no_liquid_aspirated=True)
await session.execute_phase("preparation")
display(pd.DataFrame([session.volumes()], index=["after dilution chain"]))
```

The helper checks the specific failed checkpoint, empty tracked tip and unchanged source volumes. It archives the original plan, applies the 4.3 mm preparation aspiration floor to pending transfers and retains the live connection and selected mix count. The helper issues no robot commands; `execute_phase` resumes at A32→A25 and runs only the remaining preparation. After preparation completes, continue with the refill and plate cells. This recovery does not apply if any liquid was aspirated, the tip was changed, or another operation failed.


In [ ]:
await session.execute_phase("preparation")
display(pd.DataFrame([session.volumes()], index=["after dilution chain"]))

## 5. Refill clean buffer — operator checkpoint

The last DNA-exposed tip has been discarded. Nominally A31 holds 250 µL: add **3750 µL** of the identical DNA-free buffer to restore **4000 µL TOTAL**. The physical run accepts `refilled` or `REFILLED A31` (case-insensitive) and only then updates tracking. Cancelling leaves the tracked volume unchanged and permits rerunning this checkpoint. **If you already restored A31 to 4000 µL total, do not add buffer again.**

The initial 8000 µL buffer is insufficient: dilution consumes 7500 µL and blanks require 2550 µL. This refill supplies the blank wells and leaves 1450 µL in A31. Simulation records its acknowledgement explicitly as simulated.

If an existing session is marked failed with exactly `Refill cancelled; no tracking update.`, keep the same kernel and `session`. Run the following in a separate cell; it preserves the connection and prepared samples, verifies the checkpoint, and repeats only the refill acknowledgement:

```python
import importlib
import notebook_protocol

importlib.reload(notebook_protocol)
await notebook_protocol.resume_refill(session)
print("A31 tracked total:", session.volumes()["A31"], "µL")
```

After successful acknowledgement, continue with section 6, the plate-fill cell. Do not rerun setup or preparation.

In [ ]:
await session.execute_phase("refill")
print("A31 tracked total:", session.volumes()["A31"], "µL")

## 6. Fill all 96 wells

Dispense the complete final volume directly from the manifest source into each well; no in-well dilution. Fill all blanks with one 300 µL tip, then use a separate 300 µL tip for each DNA concentration (16, 8, 4, 2, 1 and 0.5 ng/µL). Reuse that tip across all four volumes and all dispensing replicates in its group. Every well is dispensed at a fixed **2.50 mm above its inner bottom**. Tube aspirations use the verified **4 mm extra lowering**, limited by the existing 3 mm inner-bottom/reach floor. Tip contact with the destination liquid depends on its volume; the height is fixed for every well. All well assignments and replicate labels remain unchanged; the plan records actual per-well timestamps and source volumes.

Nominal final residuals: A32 250 µL; A31 1450 µL; A30–A26 225 µL each; A25 1725 µL. Plate volume is 10,200 µL. These are tracked nominal volumes; the configured total loss reserve and method validation govern physical accessibility.

This fresh run fills all 96 wells with seven 300 µL tips. The old A9 fill, B1 diagnostic aliquot, used-tip positions and 95-well continuation plan do not apply.

In [ ]:
await session.execute_phase("plate")
display(pd.DataFrame([session.volumes()], index=["final tracked source volumes"]))
print("Filled wells:", len(session.metadata["per_well_dispensing_timestamps"]))
print("Manifest, geometry, timestamps and logs:", output_dir)
await session.stop()

## 7. Manual transfer and optional F5 acquisition

Move the plate manually. Inspect for bubbles, use the same validated gentle plate handling/mixing and minimize time to reading. Remove an ordinary lid/seal from the UV light path. No gripper transfer or reader deck position is assumed.

The existing `FilterMaxF5` backend is used by `read_filtermax.py`. It reads all 96 wells, verifies the reader serial, resolves the **260 nm absorbance filter on the installed excitation slide**, and saves raw A260 plus settings, timestamps and version metadata. It does not request reference subtraction or pathlength correction.

A reader-specific plate profile (including optical read height and orientation) is separate from STARlet pipetting geometry. Complete a copy of `reader_profile.example.json` and point to the actual filter catalog/port. The optional cell remains off by default and refuses a simulated preparation record. Close other software using the reader before a separately approved reader run.

In [ ]:
RUN_READER = False
READER_PROFILE = EXPERIMENT_DIR / "reader_profile.json"
FILTER_CATALOG = EXPERIMENT_DIR.parent / "helper-scripts" / "filtermax_slides.xml"
READER_PORT = None  # enter the actual serial device path

if RUN_READER:
  from read_filtermax import load_profile, acquire
  from pylabrobot.molecular_devices.filtermax import FilterSlideCatalog
  from exp1034 import require

  require(
    not settings.simulate and "plate" in session.completed_phases,
    "Reader acquisition requires a completed physical preparation.",
  )
  require(READER_PORT is not None, "Set the actual reader port.")
  profile, reader_geometry = load_profile(READER_PROFILE)
  catalog = FilterSlideCatalog.from_softmax_xml(FILTER_CATALOG)
  import hashlib

  profile["catalog_sha256"] = hashlib.sha256(FILTER_CATALOG.read_bytes()).hexdigest()
  require(
    input("Type APPROVE F5 RUN to authorize this reader run: ").strip() == "APPROVE F5 RUN",
    "Reader run not approved.",
  )
  await acquire(
    profile,
    reader_geometry,
    catalog,
    READER_PORT,
    output_dir / "reader",
    output_dir / "preparation_record.json",
  )
else:
  print("Reader acquisition skipped; no reader connection or motion.")

## 8. Review data and analyze each volume separately

Keep the raw reader export unchanged. In a review copy, assign `valid=true/false` and a reason for each invalid/saturated reading. Retain all six individual blanks per volume. Inspect the measured calibration and choose its linear range in a copy of `calibration_ranges.example.json`; the analysis requires a documented reason and `reviewed=true`, at least three distinct concentrations and a fitted intercept.

For each volume, `analyze_a260.py` subtracts the matching blank mean, uses **sample blank SD (not SEM)** and reports:

- Screening concentration LOD = **3.3 × blank SD / slope** in ng/µL.
- Screening mass LOD = **concentration LOD × well volume** in ng.

Equal concentration does not imply equal DNA mass across volumes. The three sample wells are dispensing replicates from one chain. Repeated reader scans are reader repeats, not independent preparations. This plate does not establish a validated LOD or LOQ: confirm with fresh blanks and independently prepared low-concentration samples in later runs.

Run the analyzer after reviewing actual data:
```bash
python analyze_a260.py --input reviewed_raw_a260.csv --ranges calibration_ranges.json \
  --reader-metadata runs/PHYSICAL-RUN/reader/reader_metadata.json --output runs/analysis-01
```
The [README](README.md) contains the reagent/tip budget and data-file descriptions.